# PI4 — Ponderação por matrículas

Notebook preparatório da etapa **E04**. Compara dois modos de ler infraestrutura escolar em Guaratinguetá: percentual simples de escolas com o item versus percentual aproximado de estudantes matriculados em escolas que possuem o item.

A execução não escolhe automaticamente qual regra será final; ela produz evidência para a decisão metodológica.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime
import subprocess,sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_NAME='UNIVESP — PI4 — Infraestrutura Escolar — Guaratinguetá'
candidates=[Path('/content/drive/MyDrive')/PROJECT_NAME,Path('/content/drive/My Drive')/PROJECT_NAME]
PROJECT_ROOT=next((p for p in candidates if p.exists()),None)
if PROJECT_ROOT is None: raise FileNotFoundError('Adicione um atalho da pasta compartilhada do PI4 ao Meu Drive.')
BASE_ANALITICA=PROJECT_ROOT/'01_Dados'/'2_tratamentos_dados'/'base_analitica'


In [ ]:
!rm -rf /content/pi4_repo
!git clone -q --depth 1 https://github.com/felipecsr/univesp-projeto-integrador-4.git /content/pi4_repo
sys.path.insert(0,'/content/pi4_repo')
from src.validate_analytic_base import validate_execution
from src.eda import latest_execution_dir,load_materialized_panel,enrollment_weighted_infrastructure

REPO_COMMIT=subprocess.check_output(['git','-C','/content/pi4_repo','rev-parse','HEAD'],text=True).strip()
RUN_DIR=latest_execution_dir(BASE_ANALITICA)
reconciliacao=validate_execution(RUN_DIR)
display(reconciliacao)
assert not (reconciliacao['status']=='FAIL').any(),'P04 falhou.'
print('GATE P04: PASS')


In [ ]:
panel=load_materialized_panel(RUN_DIR)
ponderacao=enrollment_weighted_infrastructure(panel)
display(ponderacao)


## Leitura de 2025

Diferença positiva significa que estudantes estão relativamente mais concentrados em escolas que possuem o item do que sugeriria a contagem simples de escolas. Diferença negativa indica o contrário.


In [ ]:
p2025=ponderacao[ponderacao['ano']=='2025'].copy().sort_values('dif_ponderado_vs_escolas_pp')
display(p2025[['indicador','pct_escolas_com_item','pct_matriculas_em_escolas_com_item','dif_ponderado_vs_escolas_pp','escolas_validas_ponderacao','matriculas_cobertas_ponderacao']])

fig,ax=plt.subplots(figsize=(9,5.5))
ax.barh(p2025['indicador'],p2025['dif_ponderado_vs_escolas_pp'])
ax.axvline(0,linewidth=1)
ax.set_title('Guaratinguetá — efeito da ponderação por matrículas em 2025')
ax.set_xlabel('Diferença: ponderado por matrículas − % simples de escolas (p.p.)')
ax.grid(axis='x',alpha=0.2)
plt.tight_layout()
plt.show()


## Cobertura da ponderação

Escolas sem `QT_MAT_BAS` ficam fora do denominador ponderado. A tabela permite verificar se essa exclusão é material antes de decidir a regra final.


In [ ]:
cobertura=(ponderacao.groupby('ano').agg(escolas_sem_matricula=('escolas_sem_matricula','max'),matriculas_cobertas_min=('matriculas_cobertas_ponderacao','min'),matriculas_cobertas_max=('matriculas_cobertas_ponderacao','max')).reset_index())
display(cobertura)


## Materialização


In [ ]:
EDA_ROOT=PROJECT_ROOT/'01_Dados'/'2_tratamentos_dados'/'eda'
EDA_RUN=EDA_ROOT/f'ponderacao_{datetime.now().strftime("%Y%m%d_%H%M%S")}'
EDA_RUN.mkdir(parents=True,exist_ok=True)
ponderacao.to_csv(EDA_RUN/'comparacao_ponderacao_matriculas.csv',index=False,encoding='utf-8')
cobertura.to_csv(EDA_RUN/'cobertura_ponderacao.csv',index=False,encoding='utf-8')
reconciliacao.to_csv(EDA_RUN/'gate_p04.csv',index=False,encoding='utf-8')
manifest=pd.DataFrame([{'executado_em':datetime.now().isoformat(timespec='seconds'),'repo_commit':REPO_COMMIT,'base_origem':str(RUN_DIR),'regra_testada':'percentual simples de escolas vs ponderação por QT_MAT_BAS'}])
manifest.to_csv(EDA_RUN/'manifesto_ponderacao.csv',index=False,encoding='utf-8')
display(manifest)
print('Resultados gravados em:',EDA_RUN)


## Evidências manuais de E04

Guardar: **(1)** gate P04 PASS; **(2)** tabela de 2025; **(3)** gráfico das diferenças em p.p.; **(4)** cobertura da ponderação. Depois, deixar `E04M` em `Revisão`. A decisão final sobre usar ou não ponderação será tomada na revisão, não neste notebook.
